# Huffman Coding — Lossless Compression

This notebook is part of the Digital Image Processing project.

## Part B — Huffman Coding


# Huffman Coding – Lossless Compression

Huffman coding assigns prefix-free binary codes according to symbol frequency. Frequent symbols generally receive shorter codes. The encoded bit string can be decoded exactly using the same Huffman tree.

In [ ]:
import heapq
from collections import Counter
from itertools import count

class Node:
    def __init__(self, char=None, freq=0, left=None, right=None):
        self.char = char
        self.freq = freq
        self.left = left
        self.right = right

    def __lt__(self, other):
        return self.freq < other.freq

def build_huffman_tree(text):
    if not text:
        return None
    serial = count()
    heap = [(freq, next(serial), Node(char=char, freq=freq))
            for char, freq in Counter(text).items()]
    heapq.heapify(heap)

    while len(heap) > 1:
        f1, _, left = heapq.heappop(heap)
        f2, _, right = heapq.heappop(heap)
        parent = Node(freq=f1 + f2, left=left, right=right)
        heapq.heappush(heap, (parent.freq, next(serial), parent))
    return heap[0][2]

def generate_codes(root, prefix="", codes=None):
    if codes is None:
        codes = {}
    if root is None:
        return codes
    if root.char is not None:
        codes[root.char] = prefix or "0"
    else:
        generate_codes(root.left, prefix + "0", codes)
        generate_codes(root.right, prefix + "1", codes)
    return codes

def huffman_encode(text, codes):
    return "".join(codes[ch] for ch in text)

def huffman_decode(bit_string, root):
    if root is None:
        return ""
    if root.char is not None:
        if any(bit != "0" for bit in bit_string):
            raise ValueError("Invalid bit string for a single-symbol tree.")
        return root.char * len(bit_string)
    decoded = []
    node = root
    for bit in bit_string:
        if bit not in "01":
            raise ValueError("Encoded data must contain only 0 and 1.")
        node = node.left if bit == "0" else node.right
        if node is None:
            raise ValueError("Invalid encoded bit sequence.")
        if node.char is not None:
            decoded.append(node.char)
            node = root
    if node is not root:
        raise ValueError("Encoded bit sequence ended before a symbol was complete.")
    return "".join(decoded)

text = input("Enter text to compress: ")
root = build_huffman_tree(text)
codes = generate_codes(root)
encoded = huffman_encode(text, codes)
decoded = huffman_decode(encoded, root)

print("Original text:", repr(text))
print("\nCharacter frequencies:")
for char, freq in Counter(text).items():
    print(repr(char), ":", freq)
print("\nHuffman codes:")
for char, code in codes.items():
    print(repr(char), ":", code)
print("\nEncoded bit string:", encoded)
print("Decoded text:", repr(decoded))
print("Decoding successful:", decoded == text)
print("Original size (8-bit characters):", len(text) * 8, "bits")
print("Encoded size (payload only):", len(encoded), "bits")
